# Resolución de la tarea — z201_Sobre_Árboles

**Materia:** Data Mining en Economía y Finanzas · 2026
**Consigna original:** `monday/z201_Sobre_Árboles.ipynb` (Alejandro Bolaños)

Este notebook copia cada consigna tal como está en el original y, debajo, la resuelve explicando qué se busca en ese punto y cómo lo resuelvo.

Se apoya en `competencia_01_crudo.csv`. En Colab, la celda de configuración monta Drive y busca el archivo; si no lo encuentra, permite subirlo a mano.

## Configuración de Colab

Editar `DATA` si el CSV está en otra carpeta de Drive. Si no se encuentra, la celda busca `competencia_01_crudo.csv` dentro de `MyDrive/DMEyF` y, como último recurso, abre el diálogo de subida (en ese caso el parquet queda en `/content` y se pierde al cerrar la sesión).

In [ ]:
!pip install -q duckdb seaborn

In [ ]:
import pathlib
from google.colab import drive, files

drive.mount("/content/drive")

DATA = pathlib.Path("/content/drive/MyDrive/DMEyF/2026/datasets")   # <- editar si hace falta
NOMBRE = "competencia_01_crudo.csv"

if not (DATA / NOMBRE).exists():
    candidatos = list(pathlib.Path("/content/drive/MyDrive/DMEyF").rglob(NOMBRE))
    if candidatos:
        DATA = candidatos[0].parent
    else:
        print("No lo encontré en Drive: subilo a mano.")
        DATA = pathlib.Path("/content/data"); DATA.mkdir(exist_ok=True)
        subido = files.upload()
        for n, b in subido.items():
            (DATA / NOMBRE).write_bytes(b)

CRUDO = DATA / NOMBRE
PARQUET = DATA / "competencia_01.parquet"
print("usando:", CRUDO)

## Preparación del entorno

Nada acá responde a la consigna todavía: son las constantes del negocio.

Los dos números que gobiernan todo el problema:

- `ganancia_acierto = 1.072.500` — lo que vale retener a un cliente que se iba
- `costo_estimulo = 27.500` — lo que cuesta el estímulo a quien no era BAJA+2

Esa asimetría de 39 a 1 es la razón de que todo el análisis que sigue mida plata y no porcentaje de aciertos.

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree

# np.trapezoid existe desde numpy 2.0; en versiones anteriores es np.trapz
_trapz = getattr(np, "trapezoid", None) or np.trapz

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
SEMILLA = 17

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

print("crudo presente:", CRUDO.exists())

## Paso previo · el target

> Consigna de `z101_target_sql`:
>
> `, null as clase_ternaria -- Reemplazar null por la lógica que genera el target`

### Qué se busca

El dataset crudo no dice quién se fue. Solo tiene fotos mensuales: una fila por cliente y por mes. La baja hay que deducirla de la ausencia — si un cliente está en abril y no aparece en mayo, se fue.

La clase mira dos meses hacia adelante:

| clase | significado |
|---|---|
| `BAJA+1` | no está el mes que viene — ya se está yendo, es tarde |
| `BAJA+2` | está el mes que viene pero no el siguiente — este es el que hay que cazar |
| `CONTINUA` | sigue en los dos meses |

`BAJA+2` es el target real: es el único momento en que el cliente todavía está y se puede hacer algo.

### Cómo lo resuelvo

Con `lead()`, una función de ventana que mira filas futuras dentro de cada cliente. Pero antes hace falta el cross join entre clientes y períodos: si un cliente simplemente no tiene fila en un mes, `lead()` sobre la tabla original no se entera del hueco. Generando todas las combinaciones posibles, la ausencia se vuelve un `0` explícito y se puede mirar.

El detalle que hay que cuidar: para 202107 y 202108 no hay futuro que mirar — el dataset se corta ahí. Si eso no se maneja, `lead()` devuelve `null` y esas filas caen en el `else` quedando marcadas `CONTINUA`. Por eso van los dos `when ... is null then null`.

In [ ]:
if PARQUET.exists():
    print("ya está construido, sigo")
else:
    con = duckdb.connect()
    con.execute(f"create or replace table crudo as select * from read_csv_auto('{CRUDO}')")
    con.execute("""
    create or replace table competencia_01 as
    with periodos as (
        select distinct foto_mes from crudo
    ), clientes as (
        select distinct numero_de_cliente from crudo
    ), todo as (
        select numero_de_cliente, foto_mes from clientes cross join periodos
    ), flags as (
        select
            t.numero_de_cliente
          , t.foto_mes
          , c.* exclude (numero_de_cliente, foto_mes)
          , case when c.numero_de_cliente is null then 0 else 1 end as mes_0
          , lead(case when c.numero_de_cliente is null then 0 else 1 end, 1)
                over (partition by t.numero_de_cliente order by t.foto_mes) as mes_1
          , lead(case when c.numero_de_cliente is null then 0 else 1 end, 2)
                over (partition by t.numero_de_cliente order by t.foto_mes) as mes_2
        from todo t
        left join crudo c using (numero_de_cliente, foto_mes)
    )
    select
        * exclude (mes_0, mes_1, mes_2)
      , case
            when mes_1 is null then null      -- no hay mes+1 en la ventana
            when mes_1 = 0    then 'BAJA+1'
            when mes_2 is null then null      -- hay mes+1 pero no mes+2
            when mes_2 = 0    then 'BAJA+2'
            else 'CONTINUA'
        end as clase_ternaria
    from flags
    where mes_0 = 1
    """)
    con.execute(f"copy competencia_01 to '{PARQUET}' (format parquet)")
    print("construido")


def cargar(foto_mes=None, columnas="*"):
    where = f"where foto_mes = {foto_mes}" if foto_mes else ""
    return duckdb.sql(f"select {columnas} from '{PARQUET}' {where}").df()

Verificamos la cardinalidad, que es una de las preguntas del notebook original (*¿cuál es la nominalidad de cada clase?*):

In [ ]:
duckdb.sql(f"""
    select foto_mes
         , sum(clase_ternaria = 'BAJA+1')::int   as "BAJA+1"
         , sum(clase_ternaria = 'BAJA+2')::int   as "BAJA+2"
         , sum(clase_ternaria = 'CONTINUA')::int as "CONTINUA"
         , sum(clase_ternaria is null)::int      as sin_clase
         , round(100.0 * sum(clase_ternaria = 'BAJA+2') / count(*), 3) as pct_baja2
    from '{PARQUET}' group by foto_mes order by foto_mes
""").df()

**Lo que muestra:** de los 6 períodos, solo 202103–202106 son entrenables. En 202107 solo se puede asignar `BAJA+1` (hay mes+1 pero no mes+2); 202108 queda entero sin clase. Es exactamente lo correcto.

Y la respuesta a *¿cuál es la proporción del target?*: entre 0,53% y 0,70% según el mes. Un target rarísimo — y esa oscilación entre meses no es del modelo, es del mundo. Va a importar más adelante.

Trabajamos sobre 202104, como el notebook original.

In [ ]:
data = cargar(202104)
y = data["clase_ternaria"]
X = data.drop("clase_ternaria", axis=1)

print(f"{len(data):,} clientes")
print((data["clase_ternaria"].value_counts(normalize=True) * 100).round(3))

---
## Tarea 1

> **Tarea**
>
> - Cambie los parámetros del modelo y vea los impactos en cada una de las métricas.
> - ¿Cuál fue el modelo que dió más ganancia económica? ¿El peor?
> - ¿Cuál fue el modelo que dió mayor AUC? ¿El peor?
> - Calcule, dibuje y encuentre el punto de corte óptimo para: Accuracy, Sensibilidad, Especificidad, F1-Score

### Qué se busca

Que quede claro que **el árbol no clasifica: ordena.**

Si le pedimos al modelo su predicción directa, va a decir `CONTINUA` para todo el mundo — con 0,7% de eventos ninguna hoja llega a tener mayoría de `BAJA+2`. El notebook original ya lo señala: *"según el modelo, no deberíamos mandar ningún estímulo"*.

Lo que sí sirve es la **probabilidad** de `BAJA+2` de cada hoja. Con eso podemos ordenar las hojas de más riesgosa a menos, y decidir hasta dónde estimular. Cada corte posible es un modelo de negocio distinto, con su propia matriz de confusión y su propia ganancia.

Entonces la tarea tiene dos mitades:

1. **Comparar modelos** — ganancia y AUC de cada parametrización
2. **Elegir el corte** — dado un modelo, hasta qué hoja conviene estimular

### Cómo lo resuelvo

Primero tres funciones auxiliares. La primera es la del notebook original; las otras dos son las que hacen el trabajo de la tarea.

In [ ]:
def get_leaf_info(tree):
    """Lleva las hojas del árbol a una tabla: cuántos casos de cada clase cayó en cada una.

    `tree_.value` viene normalizado (proporciones), por eso se multiplica por
    `n_node_samples` para recuperar los conteos.
    """
    t = tree.tree_
    filas = []
    for i in range(t.node_count):
        if t.children_left[i] == _tree.TREE_LEAF:          # es hoja
            cuentas = t.value[i][0] * int(t.n_node_samples[i])
            fila = {"Node": i, "Samples": int(t.n_node_samples[i])}
            for j, clase in enumerate(tree.classes_):
                fila[clase] = int(round(cuentas[j]))
            filas.append(fila)
    df = pd.DataFrame(filas)
    for c in ("BAJA+1", "BAJA+2", "CONTINUA"):
        if c not in df:
            df[c] = 0
    return df


def tabla_cortes(leaf_df):
    """Ordena las hojas por P(BAJA+2) y acumula: ganancia y matriz de confusión por corte.

    Cada fila del resultado es un punto de corte posible: "estimular a los clientes
    de esta hoja y de todas las de arriba".
    """
    d = leaf_df.copy()
    # un acierto paga; cualquier otro estímulo cuesta
    d["ganancia"] = (GANANCIA_ACIERTO * d["BAJA+2"]
                     - COSTO_ESTIMULO * (d["BAJA+1"] + d["CONTINUA"]))
    d["prob_baja_2"] = d["BAJA+2"] / d["Samples"]
    d = d.sort_values("prob_baja_2", ascending=False).reset_index(drop=True)
    d["gan_acumulada"] = d["ganancia"].cumsum()

    # para las métricas binarias: BAJA+2 es el evento, todo lo demás no
    d["evento"] = d["BAJA+2"]
    d["no_evento"] = d["CONTINUA"] + d["BAJA+1"]
    total_e, total_ne = d["evento"].sum(), d["no_evento"].sum()
    d["TP"] = d["evento"].cumsum()          # eventos capturados hasta acá
    d["FP"] = d["no_evento"].cumsum()       # estimulados de más
    d["FN"] = total_e - d["TP"]             # eventos que quedaron afuera
    d["TN"] = total_ne - d["FP"]            # bien dejados afuera
    d["enviados"] = d["Samples"].cumsum()
    return d


def auc_de(d):
    """AUC por trapecios sobre la tabla de cortes.

    La tabla arranca en la primera hoja y termina en la última, así que hay que
    agregar a mano los extremos (0,0) y (1,1) — el notebook original lo advierte.
    """
    tpr = np.r_[0.0, (d["TP"] / (d["TP"].iloc[-1] + d["FN"].iloc[-1])).to_numpy(), 1.0]
    fpr = np.r_[0.0, (d["FP"] / (d["FP"].iloc[-1] + d["TN"].iloc[-1])).to_numpy(), 1.0]
    return float(_trapz(tpr, fpr)), fpr, tpr

### Las parametrizaciones a comparar

Seis configuraciones, elegidas para que cada una diga algo distinto y no por acumular. Van de la más podada a la que no tiene ningún freno:

In [ ]:
GRILLA = {
    "prof 3":                    dict(max_depth=3,    min_samples_split=80,   min_samples_leaf=1),
    "prof 5 · el de la clase":   dict(max_depth=5,    min_samples_split=80,   min_samples_leaf=1),
    "prof 8, hoja>=200":         dict(max_depth=8,    min_samples_split=1000, min_samples_leaf=200),
    "prof 12, hoja>=1":          dict(max_depth=12,   min_samples_split=80,   min_samples_leaf=1),
    "prof 12, hoja>=200":        dict(max_depth=12,   min_samples_split=1000, min_samples_leaf=200),
    "sin límites":               dict(max_depth=None, min_samples_split=2,    min_samples_leaf=1),
}

filas, curvas, tablas = [], {}, {}
for nombre, params in GRILLA.items():
    modelo = DecisionTreeClassifier(criterion="gini", random_state=SEMILLA, **params).fit(X, y)
    d = tabla_cortes(get_leaf_info(modelo))
    auc, fpr, tpr = auc_de(d)
    i = int(d["gan_acumulada"].idxmax())            # el corte que más plata deja
    filas.append({
        "modelo": nombre,
        "hojas": len(d),
        "ganancia_max": d.loc[i, "gan_acumulada"],
        "corte_prob": d.loc[i, "prob_baja_2"],
        "clientes_estimulados": int(d.loc[i, "enviados"]),
        "baja2_capturados": int(d.loc[i, "TP"]),
        "auc": auc,
    })
    curvas[nombre], tablas[nombre] = (fpr, tpr), d

res = pd.DataFrame(filas).sort_values("ganancia_max", ascending=False)
res.style.format({"ganancia_max": "${:,.0f}", "corte_prob": "{:.4f}", "auc": "{:.4f}",
                  "clientes_estimulados": "{:,.0f}"})

### ¿Cuál dio más ganancia? ¿Cuál más AUC?

**Los dos rankings dan el mismo ganador, y es el peor modelo de los seis.**

El árbol sin límites gana por ganancia y tiene AUC = 1,0000, o sea perfecto. Su ganancia es exactamente 1.139 × $1.072.500 — encontró los 1.139 `BAJA+2` de 202104, cada uno en su propia hoja, sin un solo falso positivo.

Eso es imposible fuera de estos datos. Sin tope de profundidad ni tamaño mínimo de hoja, el árbol sigue partiendo hasta aislar clientes individuales. No aprendió el patrón de la baja: aprendió quiénes son esos 1.139.

Y solo se nota porque estamos midiendo sobre los mismos datos con los que entrenó. **Un AUC de 1 en un problema así es una alarma, no un logro.**

Entre los modelos sanos los dos rankings no coinciden: `prof 12, hoja>=1` gana por ganancia, pero `prof 12, hoja>=200` tiene mejor AUC con menos hojas. Miden cosas distintas — el AUC evalúa el ordenamiento completo, la ganancia solo importa cerca del corte.

El peor de los sanos es `prof 3`: con 8 hojas no tiene resolución suficiente.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for nombre in GRILLA:
    d = tablas[nombre]
    ax[0].plot(d["enviados"], d["gan_acumulada"] / 1e6, lw=2, label=nombre)
    j = int(d["gan_acumulada"].idxmax())
    ax[0].plot(d["enviados"].iloc[j], d["gan_acumulada"].iloc[j] / 1e6, "o", ms=7, mec="white", mew=1.5)
ax[0].axhline(0, color="gray", lw=1)
ax[0].set(xlim=(0, 30_000), xlabel="clientes estimulados", ylabel="ganancia acumulada (millones $)",
          title="Ganancia acumulada según el punto de corte")
ax[0].legend(fontsize=8, frameon=False)

for nombre in GRILLA:
    f, t = curvas[nombre]
    ax[1].plot(f, t, lw=2, label=nombre)
ax[1].plot([0, 1], [0, 1], "--", color="gray", lw=1)
ax[1].set(xlabel="tasa de falsos positivos", ylabel="tasa de verdaderos positivos", title="Curvas ROC")
ax[1].legend(fontsize=8, frameon=False, loc="lower right")
plt.show()

**Lo que muestran los gráficos.** A la izquierda, todos los modelos sanos tienen un máximo interior: estimular de más destruye valor, la curva baja y cruza el cero. La línea del modelo sin límites salta al techo con 1.139 clientes y después se desploma.

A la derecha, su ROC es el ángulo recto perfecto. Cuando una ROC se ve así, la pregunta no es "qué bueno el modelo" sino "qué se filtró".

### El punto de corte y de dónde sale

Con la función de ganancia de la cátedra, un `BAJA+2` estimulado suma $G = 1.072.500$ y cualquier otro estimulado resta $C = 27.500$. Estimular a un cliente con probabilidad $p$ de ser `BAJA+2` tiene ganancia esperada

$$p \cdot G - (1-p)\cdot C > 0 \iff p > \frac{C}{G + C} = \frac{27.500}{1.100.000} = 0{,}025$$

Con 2,5% de probabilidad ya conviene estimular. No hace falta estar seguro: alcanza con que 1 de cada 40 sea baja.

In [ ]:
CORTE_TEORICO = COSTO_ESTIMULO / (GANANCIA_ACIERTO + COSTO_ESTIMULO)
print(f"corte teórico = {CORTE_TEORICO:.6f}")
print("\ncortes empíricos de cada modelo:")
print(res[["modelo", "corte_prob"]].to_string(index=False))

Los modelos sanos cortan entre 0,0253 y 0,0275 — apenas encima del valor teórico: la última hoja incluida es la de menor probabilidad que todavía supera 0,025. **La curva de ganancia no descubre el umbral: lo confirma.** Es la respuesta a la pregunta del notebook original (*¿nota alguna relación con la probabilidad de corte óptima?*).

El único que se desvía fuerte es `prof 3`, y por una razón mecánica: con 8 hojas ninguna probabilidad cae cerca de 0,025, así que corta donde puede.

### Las cuatro métricas de la consigna

#### Qué se busca

Mostrar que cada métrica pide un corte distinto, y que ninguna coincide con el que le importa al banco.

Las métricas clásicas tratan los dos tipos de error como si costaran parecido. Acá no: un falso positivo cuesta $27.500 y un falso negativo deja de ganar $1.072.500.

#### Cómo lo resuelvo

Sobre la tabla de cortes ya tenemos TP, TN, FP y FN acumulados en cada punto. Las cuatro métricas salen de ahí directo. Agrego una fila inicial que representa "no estimular a nadie", porque es un corte válido y es el que gana en dos de las cuatro.

In [ ]:
d = tablas["prof 12, hoja>=200"]

m = pd.DataFrame({"prob_baja_2": d["prob_baja_2"], "enviados": d["enviados"]})
m["sensibilidad"]  = d["TP"] / (d["TP"] + d["FN"])          # de los que se van, cuántos agarro
m["especificidad"] = d["TN"] / (d["TN"] + d["FP"])          # de los que se quedan, cuántos dejo en paz
m["precision"]     = d["TP"] / (d["TP"] + d["FP"])
m["accuracy"]      = (d["TP"] + d["TN"]) / (d["TP"] + d["TN"] + d["FP"] + d["FN"])
m["f1"] = 2 * m["precision"] * m["sensibilidad"] / (m["precision"] + m["sensibilidad"])
m["ganancia"] = d["gan_acumulada"]

# corte 0: no estimular a nadie -> todo se predice negativo
tot_e = int(d["TP"].iloc[-1] + d["FN"].iloc[-1])
tot_ne = int(d["FP"].iloc[-1] + d["TN"].iloc[-1])
m = pd.concat([pd.DataFrame([{
    "prob_baja_2": 1.0, "enviados": 0, "sensibilidad": 0.0, "especificidad": 1.0,
    "precision": np.nan, "accuracy": tot_ne / (tot_e + tot_ne), "f1": 0.0, "ganancia": 0.0,
}]), m], ignore_index=True)

optimos = pd.DataFrame([{
    "métrica": met,
    "valor óptimo": m.loc[m[met].idxmax(), met],
    "clientes a estimular": int(m.loc[m[met].idxmax(), "enviados"]),
    "corte": m.loc[m[met].idxmax(), "prob_baja_2"],
    "ganancia": m.loc[m[met].idxmax(), "ganancia"],
} for met in ("accuracy", "sensibilidad", "especificidad", "f1", "ganancia")])
optimos["% de la máxima"] = optimos["ganancia"] / m["ganancia"].max()
optimos.style.format({"valor óptimo": "{:,.4f}", "clientes a estimular": "{:,.0f}",
                      "corte": "{:.4f}", "ganancia": "${:,.0f}", "% de la máxima": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
x = m["enviados"] + 1                      # +1 para poder usar escala log

for met in ("accuracy", "sensibilidad", "especificidad", "f1"):
    estilo = dict(ls="--") if met == "accuracy" else {}   # si no, queda tapada por especificidad
    linea, = ax[0].plot(x, m[met], lw=2, label=met, **estilo)
    i = int(m[met].idxmax())
    ax[0].plot(x.iloc[i], m[met].iloc[i], "o", ms=8, color=linea.get_color(), mec="white", mew=1.5)
ax[0].set(xscale="log", ylim=(-.03, 1.05), xlabel="clientes estimulados (escala log)",
          ylabel="valor de la métrica", title="Cada métrica se maximiza en otro lado")
ax[0].legend(fontsize=9, frameon=False, loc="center left")

ax[1].plot(x, m["ganancia"] / 1e6, lw=2, color="#4a3aa7")
ax[1].axhline(0, color="gray", lw=1)
for met in ("accuracy", "sensibilidad", "especificidad", "f1"):
    i = int(m[met].idxmax())
    ax[1].plot(x.iloc[i], m["ganancia"].iloc[i] / 1e6, "o", ms=9, mec="white", mew=1.5)
i = int(m["ganancia"].idxmax())
ax[1].plot(x.iloc[i], m["ganancia"].iloc[i] / 1e6, "*", ms=18, color="#4a3aa7", mec="white", mew=1.5)
ax[1].set(xscale="log", xlabel="clientes estimulados (escala log)", ylabel="ganancia (millones $)",
          title="La ganancia que deja cada uno de esos cortes")
plt.show()

### Lo que sale de acá

- **Accuracy se maximiza no estimulando a nadie.** Con 0,70% de eventos, el clasificador "nadie se va" acierta el 99,30%. Si alguien pide optimizar accuracy, está pidiendo no hacer nada.
- **Especificidad**, igual: no estimular a nadie da especificidad perfecta.
- **Sensibilidad** se maximiza en el primer corte que captura todos los `BAJA+2` (143.317 estimulados) y pierde $2.688 millones.
- **F1** es la única con óptimo interior, y aun así deja más de la mitad de la plata: corta en 0,134, unas cinco veces más exigente que el 0,025 que pide el negocio.
- Accuracy y especificidad son casi la misma curva en este dataset — por eso accuracy va punteada. Con 0,7% de positivos, los verdaderos negativos dominan la accuracy y la vuelven una especificidad disfrazada.

**Conclusión de la tarea 1:** en un problema con clases desbalanceadas y costos asimétricos, la función de ganancia no es una métrica más. Es la única que apunta al lugar correcto.

---
## Tarea 2 · EDA

> **Tarea**
>
> Esto es solo el comienzo. Tiene que hacer un video a Miranda y para eso debe entender bien los datos. Continue con el EDA.
>
> - Sume al análisis los periodos descartados
> - ¿Se pueden construir features que den más luz a los modelos a detectar las futuras bajas?

### Cómo encaro esta parte

El notebook original propone un método, y lo sigo:

1. **Sumarizar todo**, leyéndolo junto al diccionario de datos, con valores únicos (categóricas codificadas como numéricas) y missings
2. **Dejar que el modelo elija qué mirar** — *"si las eligió el modelo, seguro nos ayudará a entender un poco más los datos, ¿no?"*
3. **Doble click en pocas variables**, con una técnica distinta según el tipo
4. **Convertir el hallazgo en una regla de negocio** y medirla en pesos
5. Y lo que agrega la consigna: **sumar los períodos descartados**

En sus gráficos las tres clases están siempre a la vista; nunca compara `BAJA+2` contra "el resto". Eso plantea dos preguntas distintas: **qué caracteriza a los enfermos**, y **cómo se distinguen los enfermos de los más enfermos** — donde `BAJA+2` es el que todavía se puede rescatar y `BAJA+1` el que ya se está yendo.

### Paso 1 · Sumarizando

El código del original, pero sobre los seis períodos.

In [ ]:
todo = cargar()          # los 6 meses
cols = [c for c in todo.columns
        if c not in ("numero_de_cliente", "foto_mes", "clase_ternaria")]

desc_stats = todo.describe(include='all')
missing_values = todo.isnull().sum()
unique_values = todo.nunique()

summary_table = pd.DataFrame({
    'Missing Values': missing_values,
    'Unique Values': unique_values
})

resumen_vars = pd.concat([summary_table.transpose(), desc_stats], axis=0).transpose()
resumen_vars.head(15)

In [ ]:
disfrazadas = resumen_vars.loc[
    [c for c in cols if resumen_vars.loc[c, "Unique Values"] <= 12], ["Unique Values", "min", "max"]
].sort_values("Unique Values")
print(f"CATEGORICAS DISFRAZADAS DE NUMERICAS ({len(disfrazadas)} de {len(cols)})")
print(disfrazadas.head(12).to_string())

con_nulos = resumen_vars.loc[[c for c in cols if resumen_vars.loc[c, "Missing Values"] > 0],
                             ["Missing Values"]]
con_nulos["pct"] = con_nulos["Missing Values"] / len(todo)
print(f"\nVARIABLES CON FALTANTES: {len(con_nulos)} de {len(cols)}")
print(con_nulos["pct"].round(4).value_counts().head(6).to_string())

**Los faltantes vienen en bloques.** Muchas variables comparten exactamente el mismo porcentaje de nulos — no es ruido disperso, son grupos de columnas que faltan juntas porque el cliente no tiene ese producto. Eso adelanta el paso 4.

### Paso 2 · Que el modelo elija qué mirar

In [ ]:
modelo_eda = DecisionTreeClassifier(criterion="gini", random_state=SEMILLA,
                                    max_depth=12, min_samples_split=1000,
                                    min_samples_leaf=200).fit(X, y)

feature_importances = pd.DataFrame({'feature': X.columns, 'importance': modelo_eda.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
feature_importances.head(10)

### Paso 3 · Doble click, una técnica por tipo

#### `ctrx_quarter` — numérica de alta cardinalidad → densidad por clase

Cantidad de transacciones del cliente en el trimestre.

In [ ]:
g = sns.FacetGrid(data, row="clase_ternaria", height=2.2, aspect=3)
g.map(sns.histplot, "ctrx_quarter", stat='density')
plt.show()

**¿Qué conclusiones saca de los gráficos?** `CONTINUA` tiene masa repartida hacia la derecha: gente que opera. Las dos clases de baja están apiladas contra el cero.

Y `BAJA+1` y `BAJA+2` tienen la misma forma. El que se va el mes que viene y el que se va en dos son, en esta variable, indistinguibles.

#### `Visa_status` — numérica de baja cardinalidad → crosstab

In [ ]:
pd.crosstab(data['Visa_status'], data['clase_ternaria'])

**¿Qué conclusiones saca de esta tabla?** La más importante no está en la tabla sino en lo que le falta: la suma de las filas no da el total de clientes. Los que no tienen tarjeta no aparecen — su `Visa_status` es nulo y el crosstab los descarta en silencio. Ese grupo invisible se mide en el paso 4.

#### `mpasivos_margen` — numérica con outliers → boxplot podado

In [ ]:
sns.boxplot(x='clase_ternaria', y='mpasivos_margen', data=data)
plt.show()

**¿Qué son esos circulitos?** Los outliers: puntos a más de 1,5 rangos intercuartiles de la caja. Unos pocos clientes con márgenes enormes estiran el eje y las cajas quedan aplastadas contra el cero. Podamos, como hace el original:

In [ ]:
lower_bound = data['mpasivos_margen'].quantile(0.05)
upper_bound = data['mpasivos_margen'].quantile(0.95)

filtered_data = data[(data['mpasivos_margen'] >= lower_bound) & (data['mpasivos_margen'] <= upper_bound)]
sns.boxplot(x='clase_ternaria', y='mpasivos_margen', data=filtered_data)
plt.show()

**¿Qué pasó en el código anterior?** Se sacó el 5% de cada cola, no para "limpiar" los datos sino para poder ver: el recorte es solo del gráfico.

**¿Qué interpretación hace?** La caja de `CONTINUA` está claramente más arriba. Y otra vez las cajas de `BAJA+1` y `BAJA+2` están una encima de la otra.

### La pregunta que los tres gráficos vienen haciendo

Tres variables, tres técnicas, el mismo resultado: los sanos se separan de los enfermos, y los enfermos no se separan entre sí. Pongámosle número con el AUC de Mann-Whitney sobre cada variable:

In [ ]:
def auc_univariada(x, y_bin):
    """AUC de Mann-Whitney sobre los valores no nulos."""
    ok = ~pd.isna(x)
    if ok.sum() < 100 or y_bin[ok].sum() < 10 or (~y_bin[ok]).sum() < 10:
        return np.nan
    r = pd.Series(x[ok]).rank().to_numpy()
    yy = y_bin[ok].astype(bool)
    n1, n0 = yy.sum(), (~yy).sum()
    return (r[yy].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

cl = data["clase_ternaria"]
enfermo = cl.isin(["BAJA+1", "BAJA+2"]).to_numpy()          # enfermo vs sano
solo_enfermos = enfermo                                     # subconjunto
mas_enfermo = (cl == "BAJA+1").to_numpy()[solo_enfermos]    # mas enfermo vs enfermo

filas = []
for c in cols:
    xv = pd.to_numeric(data[c], errors="coerce").to_numpy(float)
    filas.append({
        "variable": c,
        "sano_vs_enfermo": auc_univariada(xv, enfermo),
        "enfermo_vs_mas_enfermo": auc_univariada(xv[solo_enfermos], mas_enfermo),
    })
uni = pd.DataFrame(filas).dropna()
for c in ("sano_vs_enfermo", "enfermo_vs_mas_enfermo"):
    uni["fuerza_" + c] = (uni[c] - .5).abs()

print("QUE CARACTERIZA A LOS ENFERMOS")
print(uni.nlargest(6, "fuerza_sano_vs_enfermo")[["variable", "sano_vs_enfermo"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nQUE DISTINGUE A LOS ENFERMOS DE LOS MAS ENFERMOS")
print(uni.nlargest(6, "fuerza_enfermo_vs_mas_enfermo")[["variable", "enfermo_vs_mas_enfermo"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nvariables con |AUC-0.5| > 0.10 separando sano de enfermo      : "
      f"{(uni['fuerza_sano_vs_enfermo'] > .10).sum()} de {len(uni)}")
print(f"variables con |AUC-0.5| > 0.10 separando enfermo de mas enfermo: "
      f"{(uni['fuerza_enfermo_vs_mas_enfermo'] > .10).sum()} de {len(uni)}")

### El hallazgo

**Ninguna variable separa un `BAJA+1` de un `BAJA+2`.** La separación máxima es 0,057, contra 0,355 de la mejor variable para distinguir sano de enfermo.

El target no es una condición, es un calendario. Ningún modelo puede decir quién se va el mes que viene y quién en dos, porque los datos no lo saben. Lo máximo que se puede hacer es detectar "enfermo" y aceptar que una parte de los estimulados ya estaba perdida.

Eso justifica que la función de ganancia castigue al `BAJA+1` igual que a un `CONTINUA`: no hay forma de evitar estimularlo.

Lo único que asoma, muy débil, son variables de contacto con el banco: `thomebanking` (0,557), `ccallcenter_transacciones` (0,547), `tcallcenter` (0,545). Todas por encima de 0,5, o sea más altas en el más enfermo: antes de irse hay trámite. Pero con 0,557 es un susurro.

En dirección sano/enfermo casi todas las variables fuertes tienen AUC < 0,5: **el enfermo se apaga.**

### Paso 4 · De hallazgo a regla de negocio, medida en pesos

Reproduzco el circuito del original (lift, matriz de confusión, recall, ganancia) con `Visa_status > 0` y le agrego candidatas propias.

In [ ]:
def evaluar_regla(nombre, mascara, datos):
    """El circuito del original: lift, TP/FN/FP, recall y ganancia."""
    cl = datos["clase_ternaria"]
    mascara = mascara.fillna(False).astype(bool)
    cantidad_baja2 = (cl == "BAJA+2").sum()
    estimulados = int(mascara.sum())
    TP = int((mascara & (cl == "BAJA+2")).sum())
    FN = int(cantidad_baja2 - TP)
    FP = int(estimulados - TP)

    P_baja2 = cantidad_baja2 / datos.shape[0]
    P_regla = TP / estimulados if estimulados else 0
    return {
        "regla": nombre,
        "estimulados": estimulados,
        "TP": TP, "FN": FN, "FP": FP,
        "recall": TP / (TP + FN) if (TP + FN) else 0,
        "lift": P_regla / P_baja2 if P_baja2 else 0,
        "ganancia": TP * GANANCIA_ACIERTO - FP * COSTO_ESTIMULO,
    }

# las reglas se definen una sola vez, como funciones, para reusarlas sobre otros meses
REGLAS = {
    "Visa_status > 0  (la del original)": lambda d: d["Visa_status"] > 0,
    "sin tarjeta Visa  (Visa_status nulo)": lambda d: d["Visa_status"].isna(),
    "ctrx_quarter = 0": lambda d: d["ctrx_quarter"] == 0,
    "ctrx_quarter <= 5": lambda d: d["ctrx_quarter"] <= 5,
    "ctrx_quarter <= 20": lambda d: d["ctrx_quarter"] <= 20,
    "ctrx_quarter <= 20  Y  sin Visa": lambda d: (d["ctrx_quarter"] <= 20) & (d["Visa_status"].isna()),
    "ctrx_quarter <= 20  O  sin Visa": lambda d: (d["ctrx_quarter"] <= 20) | (d["Visa_status"].isna()),
}

reglas = (pd.DataFrame([evaluar_regla(n, f(data), data) for n, f in REGLAS.items()])
          .sort_values("ganancia", ascending=False))
reglas.style.format({"estimulados": "{:,.0f}", "TP": "{:,.0f}", "FN": "{:,.0f}", "FP": "{:,.0f}",
                     "recall": "{:.1%}", "lift": "{:.1f}x", "ganancia": "${:,.0f}"}).hide(axis="index")

### Lo que sale de comparar reglas

**La regla del original, `Visa_status > 0`, es la que menos gana:** tiene buen lift (10,7×) pero alcanza a muy pocos clientes (604) y captura solo el 4% de las bajas.

Las reglas de actividad rinden mucho mejor. Y aparece el compromiso de siempre: **lift alto no es lo mismo que ganancia alta.** `ctrx_quarter = 0` tiene el lift más alto de la tabla, pero alcanza a poquísimos clientes; conviene aflojar el corte y estimular más gente aunque el lift baje.

Ahora, la comparación contra el árbol de la tarea 1:

In [ ]:
gan_arbol = float(m["ganancia"].max())
mejor_regla = reglas.iloc[0]

print(f"mejor regla de negocio : {mejor_regla['regla']}")
print(f"  ganancia             : ${mejor_regla['ganancia']:,.0f}")
print(f"  clientes estimulados : {mejor_regla['estimulados']:,}")
print(f"  recall               : {mejor_regla['recall']:.1%}")
print(f"\narbol prof 12 / hoja>=200")
print(f"  ganancia             : ${gan_arbol:,.0f}")
print(f"  clientes estimulados : {int(m.loc[m['ganancia'].idxmax(), 'enviados']):,}")
print(f"\nla regla captura el {mejor_regla['ganancia']/gan_arbol:.0%} de lo que da el arbol,")
print("y se explica en una linea.")

Una condición de una línea captura buena parte de lo que hace el modelo. No lo reemplaza, pero sirve para explicar qué está viendo el modelo, y como piso contra el cual medir si el modelo vale la pena. (Ambos medidos in-sample en 202104.)

### Paso 5 · Sumar los períodos descartados

#### Variables que se rompen

Para cada variable y cada mes: qué porcentaje es nulo y qué porcentaje es cero. Si entre las dos suman casi 100%, esa variable no informa nada ese mes.

In [ ]:
piezas = []
for c in cols:
    piezas.append(f'avg(case when "{c}" is null then 1.0 else 0.0 end) as "nul__{c}"')
    piezas.append(f'avg(case when "{c}" = 0 then 1.0 else 0.0 end) as "cero__{c}"')

perfil = duckdb.sql(f"select foto_mes, {', '.join(piezas)} from '{PARQUET}' "
                    f"group by foto_mes order by foto_mes").df().set_index("foto_mes")

nulos = perfil[[c for c in perfil if c.startswith("nul__")]].rename(columns=lambda s: s[5:])
ceros = perfil[[c for c in perfil if c.startswith("cero__")]].rename(columns=lambda s: s[6:])
muerta = (nulos + ceros) >= 0.999

rotas = muerta.columns[muerta.any() & ~muerta.all()]
print("VARIABLES QUE SE ROMPEN EN ALGUN MES\n")
print(muerta[rotas].T.replace({True: "ROTA", False: "ok"}).to_string())
print("\nMuertas en los 6 meses (descartables):", list(muerta.columns[muerta.all()]))

`mpayroll2` y `cpayroll2_trx` tienen información solo en 202106. Un modelo entrenado ahí aprende un patrón que en producción no va a estar. `ccajas_depositos` se cae justo en 202105.

**Esto es data drifting**, y es el tipo de cosa que hunde un modelo en el privado sin que se note en el público.

#### ¿La regla de negocio aguanta todos los meses?

In [ ]:
nombre_mejor = reglas.iloc[0]["regla"]
condicion_mejor = REGLAS[nombre_mejor]          # misma regla, no una copia a mano

filas = []
for mes in (202103, 202104, 202105, 202106):     # los 4 meses con clase completa
    dm = cargar(mes)
    r = evaluar_regla(nombre_mejor, condicion_mejor(dm), dm)
    r["foto_mes"] = mes
    r["pct_estimulados"] = r["estimulados"] / len(dm)
    r["baja2_del_mes"] = int((dm["clase_ternaria"] == "BAJA+2").sum())
    filas.append(r)

print(f"regla evaluada: {nombre_mejor}\n")
estabilidad = pd.DataFrame(filas)[["foto_mes", "baja2_del_mes", "estimulados", "pct_estimulados",
                                   "TP", "recall", "lift", "ganancia"]]
estabilidad.style.format({"baja2_del_mes": "{:,.0f}", "estimulados": "{:,.0f}",
                          "pct_estimulados": "{:.1%}", "TP": "{:,.0f}", "recall": "{:.1%}",
                          "lift": "{:.1f}x", "ganancia": "${:,.0f}"}).hide(axis="index")

### Lo que muestra

**El alcance de la regla es estable, la plata que deja no.**

La regla toca siempre entre el 7,1% y el 7,6% de los clientes, y el lift se mueve entre 5,9× y 7,4×. Nada de eso es alarmante.

Pero la ganancia va de \$82,5M (202105) a \$356,7M (202104): un factor de 4,3. Y eso no se explica solo por la cantidad de bajas del mes (870 contra 1.139, un 31% de diferencia).

La ganancia es una resta entre dos números grandes. Con unos 12.000 estimulados por mes y alrededor de 11.500 falsos positivos, el costo es prácticamente fijo en unos \$320M:

`ganancia ≈ TP × $1.072.500 − ~$320.000.000`

En 202105 la regla capturó 374 bajas y en 202104, 625. Esa diferencia de 251 clientes, a un millón cada uno, es casi toda la brecha. Un cambio moderado en los aciertos hace saltar el resultado.

**La regla no falla, pero su resultado es frágil.** Y es el mismo fenómeno que va a aparecer en z301: buena parte de la variación entre meses no es del modelo ni de la regla, es del mes.

### Paso 6 · ¿Se pueden construir features?

#### Qué se busca

El paso 3 da la pista: **el cliente se apaga**. Si es así, lo que debería predecir no es el nivel de actividad sino la caída — un cliente que siempre hizo 5 transacciones no es lo mismo que uno que hacía 50 y ahora hace 5. Eso el modelo no lo puede ver mirando la foto de un mes.

#### Cómo lo resuelvo

Con funciones de ventana, las mismas de z101, sobre las variables:

- `lag1` — el valor del mes pasado
- `delta1` — la variación contra el mes pasado
- `vs_hist` — el valor de hoy contra su propio promedio de los 3 meses previos (<1: por debajo de lo habitual para ese cliente)
- `sin_visa` / `sin_master` / `tarjetas_faltantes` — el hallazgo de los nulos (pasos 3 y 4), hecho variable

`vs_hist` es un ratio, así que la división por cero va con `nullif`.

In [ ]:
CLAVE = ["ctrx_quarter", "mcaja_ahorro", "mpasivos_margen", "mtarjeta_visa_consumo",
         "mcuentas_saldo", "cproductos", "mpayroll", "chomebanking_transacciones",
         "mrentabilidad", "mactivos_margen"]
ROTAS = ["mpayroll2", "cpayroll2_trx", "ccajas_depositos",
         "mcuenta_corriente_adicional", "Master_madelantodolares", "Visa_madelantodolares"]

base_cols = [c for c in cols if c not in ROTAS]
vent = "over (partition by numero_de_cliente order by foto_mes"

nuevas = []
for v in CLAVE:
    nuevas += [
        f'lag("{v}", 1) {vent}) as "{v}__lag1"',
        f'"{v}" - lag("{v}", 1) {vent}) as "{v}__delta1"',
        f'"{v}" / nullif(avg("{v}") {vent} rows between 3 preceding and 1 preceding), 0) as "{v}__vs_hist"',
    ]
nuevas += [
    "case when Visa_status is null then 1 else 0 end as sin_visa",
    "case when Master_status is null then 1 else 0 end as sin_master",
    "(case when Visa_status is null then 1 else 0 end + "
    " case when Master_status is null then 1 else 0 end) as tarjetas_faltantes",
]

con_hist = duckdb.sql(f"""
    with h as (
        select numero_de_cliente, foto_mes, clase_ternaria
             , {', '.join(f'"{c}"' for c in base_cols)}
             , {', '.join(nuevas)}
        from '{PARQUET}'
    ) select * from h where foto_mes = 202104
""").df()

nuevas_cols = [c for c in con_hist.columns
               if c not in base_cols + ["numero_de_cliente", "foto_mes", "clase_ternaria"]]
print(f"{len(base_cols)} variables originales + {len(nuevas_cols)} construidas")

In [ ]:
usar = base_cols + nuevas_cols
mod = DecisionTreeClassifier(criterion="gini", random_state=SEMILLA, max_depth=12,
                             min_samples_split=1000, min_samples_leaf=200
                             ).fit(con_hist[usar], con_hist["clase_ternaria"])

imp = (pd.DataFrame({"variable": usar, "importancia": mod.feature_importances_})
       .query("importancia > 0").sort_values("importancia", ascending=False))
imp["construida"] = np.where(imp["variable"].isin(nuevas_cols), "sí", "")

print(f"las {len(nuevas_cols)} features construidas se llevan el "
      f"{imp.loc[imp.construida == 'sí', 'importancia'].sum():.1%} de la importancia total\n")
imp.head(12).style.format({"importancia": "{:.4f}"}).hide(axis="index")

### Lo que muestra, y lo que todavía no se puede saber

El árbol usa algunas de las construidas — sobre todo `tarjetas_faltantes` y algunos `vs_hist`.

Pero **que el árbol las use no significa que mejoren nada.** Toda la medición de este notebook está hecha sobre los mismos datos de entrenamiento — el mismo mecanismo que hizo que el árbol sin límites pareciera perfecto en la tarea 1.

Para saber si ayudan hay que medir la ganancia en un mes que el modelo no vio. Eso es `z301_Sobre_la_incertidumbre`, y por eso queda planteado y no resuelto acá. (El script `06_features.py` tiene esa medición: con un solo árbol, las features de historia no mejoran nada — −1,3% en un mes, +1,6% en otro: ruido. El único aporte real es `tarjetas_faltantes`.)

---
## Resumen

### Tarea 1

| pregunta | respuesta |
|---|---|
| ¿Más ganancia? | El árbol sin límites: \$1.221.577.500 — pero porque memorizó |
| ¿Peor ganancia? | `prof 3`, con 8 hojas: \$363.962.500 |
| ¿Mayor AUC? | El mismo sin límites: 1,0000 — la señal de alarma |
| ¿Peor AUC? | `prof 3`: 0,8127 |
| Mejor modelo real | `prof 12, hoja>=200` — mejor AUC entre los sanos, con menos hojas |

Punto de corte óptimo por métrica (`prof 12, hoja>=200`):

| métrica | clientes a estimular | ganancia |
|---|---|---|
| accuracy | 0 | \$0 |
| especificidad | 0 | \$0 |
| sensibilidad | 143.317 | −\$2.688.317.500 |
| F1 | 1.324 | 45,5% de la máxima |
| ganancia | 12.832 | \$548.020.000 |

### Tarea 2

- `mpayroll2` y `cpayroll2_trx` solo informan en 202106 — data drifting
- No tener tarjeta Visa: lift 5,5×. El nulo es señal; imputarlo la borra
- Sin transacciones en el trimestre: lift 14,4×
- Ninguna variable separa `BAJA+1` de `BAJA+2`: el target es un calendario
- Las features de historia se pueden construir; para saber si sirven falta z301

### Para el video de Miranda

- El cliente se apaga antes de irse — hay ventana para actuar
- No tener tarjeta Visa multiplica por 5,5 el riesgo, y es un flag, no un modelo
- La campaña conviene con p > 2,5%, y eso sale de la aritmética del negocio
- Optimizar accuracy sería no llamar a nadie